In [ ]:
import json
import shutil
from pathlib import Path
import torch

work = Path('/kaggle/working')
if not work.is_dir():
    raise RuntimeError('Run this cell inside the Kaggle notebook.')
report = {'torch': torch.__version__, 'cuda_runtime': torch.version.cuda,
          'disk_free_gib': round(shutil.disk_usage(work).free / 2**30, 2), 'gpus': []}
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    with torch.cuda.device(i):
        x = torch.tensor([2., 3.], device=f'cuda:{i}', requires_grad=True)
        x.square().sum().backward()
        passed = x.grad.tolist() == [4., 6.]
    report['gpus'].append({'name': props.name, 'vram_gib': round(props.total_memory / 2**30, 2),
                           'backward_pass': passed})
print(json.dumps(report, indent=2))
(work / 'neurobrain_gpu_check.json').write_text(json.dumps(report, indent=2))
if not report['gpus'] or not all(g['backward_pass'] for g in report['gpus']):
    raise RuntimeError('GPU preflight failed. Check the accelerator setting.')
